# Model Training and Evaluation

This notebook demonstrates the complete workflow for training and evaluating computer vision models.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys

sys.path.append('..')

from src.models.train import CVModel, ModelTrainer
from src.models.evaluate import ModelEvaluator
from src.models.predict import ModelInference
from src.utils.config import Config
from src.utils.helpers import set_seed
from src.visualization.visualize import plot_training_history, plot_confusion_matrix

# Set random seed
set_seed(42)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Configuration

In [ ]:
# Load configuration
config = Config('../config.yaml')

print("Configuration:")
print(f"  Model: {config.get('model.architecture')}")
print(f"  Number of classes: {config.get('model.num_classes')}")
print(f"  Learning rate: {config.get('training.learning_rate')}")
print(f"  Epochs: {config.get('training.epochs')}")
print(f"  Batch size: {config.get('data.batch_size')}")

## 2. Initialize Model

In [ ]:
# Initialize model
model = CVModel(
    model_name=config.get('model.architecture'),
    num_classes=config.get('model.num_classes'),
    pretrained=config.get('model.pretrained')
)

# Count parameters
from src.utils.helpers import count_parameters
num_params = count_parameters(model)
print(f"Model initialized with {num_params:,} trainable parameters")

## 3. Prepare Data Loaders

**Note**: This is a placeholder. You need to implement actual data loading based on your dataset format.

In [ ]:
# Create dummy data for demonstration
# Replace this with actual data loading

batch_size = config.get('data.batch_size')
num_classes = config.get('model.num_classes')

# Dummy training data
X_train = torch.randn(100, 3, 224, 224)
y_train = torch.randint(0, num_classes, (100,))
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Dummy validation data
X_val = torch.randn(30, 3, 224, 224)
y_val = torch.randint(0, num_classes, (30,))
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batch size: {batch_size}")

## 4. Train Model

In [ ]:
# Initialize trainer
trainer = ModelTrainer(model)

# Train model
print("Starting training...")
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,  # Reduced for demonstration
    lr=config.get('training.learning_rate'),
    save_path='../models/saved_models'
)

print("\nTraining completed!")
print(f"Best validation accuracy: {history['best_val_accuracy']:.2f}%")

## 5. Visualize Training History

In [ ]:
# Plot training history
plot_training_history(history)
plt.show()

## 6. Evaluate Model

In [ ]:
# Load best model for evaluation
evaluator = ModelEvaluator(model)
evaluator.load_model('../models/saved_models/best_model.pth')

# Evaluate on validation set
results = evaluator.evaluate(val_loader)

print("\nEvaluation Metrics:")
print(f"  Accuracy: {results['metrics']['accuracy']:.4f}")
print(f"  Precision (macro): {results['metrics']['precision_macro']:.4f}")
print(f"  Recall (macro): {results['metrics']['recall_macro']:.4f}")
print(f"  F1-score (macro): {results['metrics']['f1_macro']:.4f}")

## 7. Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = np.array(results['confusion_matrix'])
class_names = [f"Class {i}" for i in range(num_classes)]

plot_confusion_matrix(cm, class_names)
plt.show()

## 8. Make Predictions

In [ ]:
# Initialize inference engine
inference = ModelInference(
    model,
    model_path='../models/saved_models/best_model.pth',
    class_names=class_names
)

# Make predictions on sample images
sample_image = (X_val[0].numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
sample_image = np.ascontiguousarray(sample_image)

prediction = inference.predict(sample_image, top_k=5)

print("\nTop 5 Predictions:")
for i, pred in enumerate(prediction['predictions'], 1):
    print(f"  {i}. {pred['class_name']}: {pred['probability']:.2%}")

## 9. Save Results

In [ ]:
# Save evaluation results
evaluator.save_results(results, '../results/evaluation_results.json')

# Save training history plot
plot_training_history(history, save_path='../results/training_history.png')

# Save confusion matrix
evaluator.plot_confusion_matrix(cm, class_names, save_path='../results/confusion_matrix.png')

print("Results saved successfully!")

## Summary

This notebook demonstrated:
1. Model initialization and configuration
2. Training a computer vision model
3. Evaluating model performance
4. Visualizing training metrics and confusion matrix
5. Making predictions on new images

Next steps:
- Experiment with different model architectures
- Tune hyperparameters
- Apply data augmentation techniques
- Deploy the model using Flask API or Docker